## Requires Python 3.12

## JobSpy Job Scraper — Munich Working Student (Data Scientist / Analyst) Roles

This notebook is **self-healing and self-resolving**: it installs JobSpy, patches a
known Glassdoor bug, then **automatically tries multiple location and search-term
variants per platform** until it finds one that returns results — so you don't have
to manually guess the exact string each job board expects.

**What it does:**
1. Installs/upgrades `python-jobspy` (with fallback for externally-managed Python)
2. Monkey-patches the Glassdoor location-lookup bug at runtime (no manual file editing)
3. **Auto-resolves location + search term per platform** — tries a list of candidate
   variants (e.g. `["München", "Bayern"]`, `["Munich", "Bavaria"]`) and keeps the
   first one that returns at least one job
4. Runs Google Jobs separately, since it requires a different query format entirely
5. Merges everything into one DataFrame with a guaranteed `description` column
6. Saves results to `jobs.csv`


## Python Installs and Fixes

In [1]:
# Install / upgrade JobSpy. Falls back automatically on "externally managed
# environment" errors (common on some system Python installs on Linux/Mac;
# not an issue inside a normal Windows venv).
import subprocess, sys

def pip_install(*args):
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-U", *args],
        capture_output=True, text=True,
    )
    if result.returncode != 0 and "externally-managed-environment" in result.stderr:
        print("System Python is externally managed — retrying with --break-system-packages")
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-U", "--break-system-packages", *args],
            capture_output=True, text=True,
        )
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"pip install failed for: {args}")
    installed_names = " ".join(args)
    print(f"Installed/updated: {installed_names}")

pip_install("python-jobspy")


Installed/updated: python-jobspy


#### Auto-patch the Glassdoor location bug

As of `python-jobspy` 1.1.82 (and the unmerged GitHub PR #347), the Glassdoor
scraper builds its location-lookup URL with an f-string and **never
URL-encodes the location string**:

```python
url = f"{self.base_url}/findPopularLocationAjax.htm?maxLocationsToReturn=10&term={location}"
```

Any location with a comma, space, or non-ASCII character (e.g. `"München, Bayern"`)
gets sent raw, which Glassdoor's endpoint rejects with HTTP 400 → `"location not parsed"`.

Rather than editing the installed package file by hand (which breaks on the next
`pip install`/on a different machine), the cell below **monkey-patches the method
at runtime, in memory**, every time the notebook runs. This requires no
filesystem edits and survives reinstalls/upgrades/other machines.


In [2]:
import inspect
import urllib.parse
import jobspy.glassdoor as gd_module

def _patched_get_location(self, location: str, is_remote: bool):
    """Drop-in replacement for Glassdoor._get_location with URL-encoding fixed."""
    if not location or is_remote:
        return "11047", "STATE"  # remote options
    term = urllib.parse.quote(location)
    url = f"{self.base_url}/findPopularLocationAjax.htm?maxLocationsToReturn=10&term={term}"
    res = self.session.get(url)
    if res.status_code != 200:
        if res.status_code == 429:
            err = "429 Response - Blocked by Glassdoor for too many requests"
        else:
            err = f"Glassdoor response status code {res.status_code}"
        gd_module.log.error(err)
        return None, None
    items = res.json()
    if not items:
        raise ValueError(f"Location \'{location}\' not found on Glassdoor")
    location_type = items[0]["locationType"]
    if location_type == "C":
        location_type = "CITY"
    elif location_type == "S":
        location_type = "STATE"
    elif location_type == "N":
        location_type = "COUNTRY"
    return int(items[0]["locationId"]), location_type

_patched_get_location._is_jobspy_location_patch = True


def patch_glassdoor_location_bug():
    """Apply the runtime patch only if the installed version still has the bug.
    Safe to call multiple times (idempotent) and safe once JobSpy ships a real fix
    upstream (it self-detects and skips)."""
    current = gd_module.Glassdoor._get_location
    if getattr(current, "_is_jobspy_location_patch", False):
        print("Patch already applied this session — skipping.")
        return False
    try:
        source = inspect.getsource(current)
        if "quote(" in source:
            print("JobSpy already fixed upstream — no patch needed.")
            return False
    except OSError:
        pass  # can\'t introspect source (e.g. already patched in a prior cell run) — proceed
    gd_module.Glassdoor._get_location = _patched_get_location
    print("Patched Glassdoor._get_location (URL-encoding fix applied).")
    return True

patch_glassdoor_location_bug()


Patched Glassdoor._get_location (URL-encoding fix applied).


True

## Search - Config
Different job boards index locations and parse search terms differently — what
works on Glassdoor (`"München, Bayern"`) may not work on Indeed, and vice versa.
Rather than hardcoding one string and hoping, define an **ordered list of
candidate variants** per category. The resolver below tries each in order and
keeps the first one that returns at least one job, for each site independently.

Edit these lists for a different city/role/country — everything downstream
adapts automatically.


In [ ]:
!ipconfig /flushdns


Windows-IP-Konfiguration

Der DNS-Aufl�sungscache wurde geleert.
